<a href="https://colab.research.google.com/github/lianghuizi/Study/blob/main/2026_08_14_PyTorch_MHA%E6%BA%90%E7%A0%81%E5%AE%9A%E4%BD%8D_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch MultiheadAttention 源码定位（Colab版）\n\n本 Notebook 不需要本地环境或 GPU；在 Colab CPU 中从上到下运行。

In [2]:
import sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Python: 3.12.13
PyTorch: 2.11.0+cpu
CUDA available: False


In [3]:
import inspect
import torch.nn.functional as F
from torch import nn
mha_file=inspect.getsourcefile(nn.MultiheadAttention)
mha_line=inspect.getsourcelines(nn.MultiheadAttention.forward)[1]
functional_file=inspect.getsourcefile(F.multi_head_attention_forward)
functional_line=inspect.getsourcelines(F.multi_head_attention_forward)[1]
print(mha_file, mha_line)
print(functional_file, functional_line)
print(inspect.getsource(nn.MultiheadAttention.forward))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py 1255
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py 6244
    def forward(
        self,
        query: Tensor,
        key: Tensor,
        value: Tensor,
        key_padding_mask: Tensor | None = None,
        need_weights: bool = True,
        attn_mask: Tensor | None = None,
        average_attn_weights: bool = True,
        is_causal: bool = False,
    ) -> tuple[Tensor, Tensor | None]:
        r"""Compute attention outputs using query, key, and value embeddings.

            Supports optional parameters for padding, masks and attention weights.

        Args:
            query: Query embeddings of shape :math:`(L, E_q)` for unbatched input, :math:`(L, N, E_q)` when ``batch_first=False``
                or :math:`(N, L, E_q)` when ``batch_first=True``, where :math:`L` is the target sequence length,
                :math:`N` is the batch size, and :math:`E_q` is the query embedding dimension ``emb

In [5]:
batch_size,seq_len,embed_dim,num_heads=2,5,32,4
head_dim=embed_dim//num_heads
mha=nn.MultiheadAttention(embed_dim,num_heads,batch_first=True)
x=torch.randn(batch_size,seq_len,embed_dim)
with torch.no_grad():
  output,weights=mha(x,x,x,need_weights=True,average_attn_weights=False)
print('input:',tuple(x.shape))
print('head_dim:',head_dim)
print('output:',tuple(output.shape))
print('weights:',tuple(weights.shape))
assert output.shape==(2,5,32) and weights.shape==(2,4,5,5)
print('✅ shape checks passed')

input: (2, 5, 32)
head_dim: 8
output: (2, 5, 32)
weights: (2, 4, 5, 5)
✅ shape checks passed


In [7]:
note=f'''# MHA 调用链

- MultiheadAttention.forward: {mha_file}:{mha_line}
- multi_head_attention_forward: {functional_file}:{functional_line}
- 输入 [B,L,E]=[2,5,32]
- 拆 head 后 [B,H,L,D]=[2,4,5,8]
- weights [B,H,L,L]=[2,4,5,5]

调用链：用户代码 → __call__ → MultiheadAttention.forward → functional → QKV projection → reshape/transpose → scaled dot-product attention → concat/output projection'''
open('/content/mha-call-chain.md','w',encoding='utf-8').write(note)
from google.colab import files
files.download('/content/mha-call-chain.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>